# Loan Default Risk Model Training

This notebook trains two machine learning models for loan default risk assessment:

1. **Behavioral Classification Model** (Random Forest): Classifies users based on transaction behavior patterns
2. **Default Risk Prediction Model** (XGBoost): Predicts probability of loan default

Both models are logged with MLflow and registered to Unity Catalog for production deployment.

## 1. Setup and Data Loading

Import required libraries and load data from Databricks tables.

In [0]:
%pip install xgboost scikit-learn mlflow --quiet
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, confusion_matrix, roc_auc_score
)
import xgboost as xgb
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from mlflow.models.signature import infer_signature
import warnings

warnings.filterwarnings('ignore')
print("Libraries imported successfully")

In [0]:
# Load transactions data
print("Loading transactions data...")
df = spark.table("hackathon.raw.transactions").toPandas()
df['date'] = pd.to_datetime(df['date'])
print(f"  Transactions: {len(df):,} rows")

# Load loan data
print("Loading loan data...")
loans = spark.table("hackathon.raw.loan_data").toPandas()
print(f"  Loan records: {len(loans):,} rows")

print("\nData loaded successfully!")

## 2. Behavioral Feature Engineering

Extract behavioral features from transaction data to understand user spending patterns, income stability, and financial health.

In [0]:
print("Engineering behavioral features...")

# Basic user aggregations
user_agg = df.groupby("user_id").agg(
    total_transactions=('amount', 'count'),
    avg_transaction_amount=('amount', 'mean'),
    avg_balance=('balance', 'mean'),
    min_balance=('balance', 'min'),
    max_balance=('balance', 'max'),
    balance_std=('balance', 'std')
).fillna(0)

print(f"Created base aggregations for {len(user_agg):,} users")

In [0]:
# Conditional features
is_credit = df['transaction_type'] == 'credit'
is_debit = df['transaction_type'] == 'debit'

df['is_small_credit'] = is_credit & (df['amount'] < 5000)
df['is_large_debit'] = is_debit & (df['amount'] > 10000)

cond_agg = df.groupby("user_id").agg(
    small_credit_count=('is_small_credit', 'sum'),
    large_debit_count=('is_large_debit', 'sum'),
    cash_count=('channel', lambda x: (x == 'cash').sum()),
    upi_count=('channel', lambda x: (x.str.contains('UPI', case=False, na=False)).sum()),
    label_user_type=('label_user_type', 'last')
)

cond_agg['cash_vs_upi_ratio'] = cond_agg['cash_count'] / cond_agg['upi_count'].replace(0, 1)
cond_agg = cond_agg.drop(columns=['cash_count', 'upi_count'])

print("Created conditional aggregations")

In [0]:
# Monthly metrics
df['year_month'] = df['date'].dt.to_period('M')

monthly_inflow = df[is_credit].groupby(['user_id', 'year_month'])['amount'].sum().reset_index(name='m_inflow')
monthly_outflow = df[is_debit].groupby(['user_id', 'year_month'])['amount'].sum().reset_index(name='m_outflow')
monthly_txns = df.groupby(['user_id', 'year_month']).size().reset_index(name='m_txns')

monthly = pd.merge(monthly_txns, monthly_inflow, on=['user_id', 'year_month'], how='left')
monthly = pd.merge(monthly, monthly_outflow, on=['user_id', 'year_month'], how='left').fillna(0)

monthly_agg = monthly.groupby('user_id').agg(
    monthly_inflow=('m_inflow', 'mean'),
    monthly_outflow=('m_outflow', 'mean'),
    inflow_variance=('m_inflow', 'var'),
    txn_frequency=('m_txns', 'mean')
).fillna(0)

monthly_agg['inflow_outflow_ratio'] = monthly_agg['monthly_inflow'] / monthly_agg['monthly_outflow'].replace(0, 1)

print("Created monthly aggregations")

In [0]:
# Credit-debit time gap
df_sorted = df.sort_values(['user_id', 'date'])
df_sorted['credit_time'] = pd.NaT
is_credit_sorted = df_sorted['transaction_type'] == 'credit'
df_sorted.loc[is_credit_sorted, 'credit_time'] = df_sorted.loc[is_credit_sorted, 'date']
df_sorted['last_credit_time'] = df_sorted.groupby('user_id')['credit_time'].ffill()

df_debits = df_sorted[df_sorted['transaction_type'] == 'debit'].copy()
df_debits['time_gap_hrs'] = (df_debits['date'] - df_debits['last_credit_time']).dt.total_seconds() / 3600.0

gap_agg = df_debits.groupby('user_id').agg(
    credit_debit_time_gap=('time_gap_hrs', 'mean')
).fillna(0)

print("Created time gap features")

In [0]:
# Consecutive zero balance days
df_sorted['date_day'] = df_sorted['date'].dt.date
daily_balance = df_sorted.drop_duplicates(subset=['user_id', 'date_day'], keep='last').copy()
daily_balance['is_zero'] = daily_balance['balance'] < 1000
daily_balance['island'] = (~daily_balance['is_zero']).groupby(daily_balance['user_id']).cumsum()

island_counts = daily_balance[daily_balance['is_zero']].groupby(['user_id', 'island']).size()
if len(island_counts) > 0:
    zero_agg = island_counts.groupby('user_id').max().rename('consecutive_zero_balance_days').to_frame()
else:
    zero_agg = pd.DataFrame(columns=['consecutive_zero_balance_days'])

print("Created balance streak features")

In [0]:
# Combine all behavioral features
behavioral_features = user_agg.join(cond_agg).join(monthly_agg).join(gap_agg).join(zero_agg).fillna(0)

print(f"\nBehavioral feature engineering completed for {len(behavioral_features):,} users")
print(f"Total behavioral features: {len(behavioral_features.columns)}")
display(behavioral_features.head())

## 3. Debt Feature Engineering

Extract debt-related features from loan data to understand repayment behavior and default risk.

In [0]:
print("Engineering debt features...")

# Aggregate loan data by user
loan_agg = loans.groupby("user_id").agg(
    total_loan_amount=('loan_amount', 'sum'),
    total_amount_repaid=('amount_repaid', 'sum'),
    total_outstanding=('outstanding_amount', 'sum'),
    missed_payment_count=('missed_payments', 'sum'),
    avg_days_past_due=('days_past_due', 'mean'),
    max_days_past_due=('days_past_due', 'max'),
    avg_interest_rate=('interest_rate', 'mean'),
    num_loans=('loan_id', 'count'),
    default_flag=('default_flag', 'max'),  # user-level: defaulted if any loan defaulted
)

print(f"Created loan aggregations for {len(loan_agg):,} users")

In [0]:
# Merge behavioral + debt features
final_features = behavioral_features.join(loan_agg, how='left').fillna(0)

# Derived debt features
final_features['debt_to_income_ratio'] = final_features['total_loan_amount'] / final_features['monthly_inflow'].replace(0, 1) / 12
final_features['repayment_ratio'] = final_features['total_amount_repaid'] / final_features['total_loan_amount'].replace(0, 1)
final_features['outstanding_to_balance_ratio'] = final_features['total_outstanding'] / final_features['avg_balance'].replace(0, 1)

# Debt growth rate
net_owed = final_features['total_loan_amount'] - final_features['total_amount_repaid']
final_features['debt_growth_rate'] = (final_features['total_outstanding'] - net_owed) / net_owed.replace(0, 1)

print(f"\nFinal feature set created with {len(final_features.columns)} features for {len(final_features):,} users")
display(final_features.head())

## 4. Model 1: Behavioral Classification (Random Forest)

Classify users into behavioral categories based on transaction patterns.

In [0]:
# Define behavioral feature columns
behavioral_cols = [
    "total_transactions", "txn_frequency", "avg_transaction_amount",
    "monthly_inflow", "monthly_outflow", "inflow_outflow_ratio", "inflow_variance",
    "avg_balance", "min_balance", "max_balance", "balance_std",
    "small_credit_count", "large_debit_count", "credit_debit_time_gap",
    "consecutive_zero_balance_days", "cash_vs_upi_ratio"
]

X_behav = final_features[behavioral_cols]
y_behav = final_features['label_user_type']

print(f"Features shape: {X_behav.shape}")
print(f"Target classes: {y_behav.unique()}")
print(f"Class distribution:\n{y_behav.value_counts()}")

In [0]:
# Split with stratification
X_b_train, X_b_test, y_b_train, y_b_test = train_test_split(
    X_behav, y_behav, test_size=0.2, stratify=y_behav, random_state=42
)

print(f"Training set: {len(X_b_train):,} samples")
print(f"Test set: {len(X_b_test):,} samples")

In [0]:
print("Training Random Forest Classifier...")

# Start MLflow run for behavioral classifier
mlflow.set_experiment("/Users/arnav.prasad999918@gmail.com/loan-default-risk")

with mlflow.start_run(run_name="behavioral_classifier") as run:
    # Train model
    rf = RandomForestClassifier(
        n_estimators=100, 
        max_depth=10, 
        random_state=42, 
        class_weight='balanced'
    )
    rf.fit(X_b_train, y_b_train)
    
    # Predictions
    y_b_pred = rf.predict(X_b_test)
    
    # Calculate metrics
    acc = accuracy_score(y_b_test, y_b_pred)
    prec = precision_score(y_b_test, y_b_pred, average='weighted')
    rec = recall_score(y_b_test, y_b_pred, average='weighted')
    f1 = f1_score(y_b_test, y_b_pred, average='weighted')
    
    # Log parameters and metrics
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)
    mlflow.log_metric("f1_score", f1)
    
    # Feature importances
    feature_importance = pd.DataFrame({
        'feature': behavioral_cols,
        'importance': rf.feature_importances_
    }).sort_values('importance', ascending=False)
    
    # Log model with signature
    signature = infer_signature(X_b_train, y_b_train)
    mlflow.sklearn.log_model(
        rf, 
        "model",
        signature=signature,
        input_example=X_b_train.head(5),
        registered_model_name="hackathon.default.behavioral_classifier"
    )
    
    print("\n" + "="*55)
    print("  BEHAVIORAL CLASSIFICATION RESULTS")
    print("="*55)
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"\nMLflow Run ID: {run.info.run_id}")

In [0]:
# Display confusion matrix
print("\nConfusion Matrix:")
labels = rf.classes_
cm = confusion_matrix(y_b_test, y_b_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
display(cm_df)

In [0]:
# Display feature importances
print("Feature Importances (Top 10):")
feature_importance = pd.DataFrame({
    'feature': behavioral_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

display(feature_importance.head(10))

## 5. Model 2: Default Prediction (XGBoost)

Predict loan default probability using both behavioral and debt features.

In [0]:
# Define all feature columns (behavioral + debt)
default_cols = behavioral_cols + [
    "debt_to_income_ratio", "repayment_ratio", "missed_payment_count",
    "avg_days_past_due", "debt_growth_rate", "outstanding_to_balance_ratio",
    "total_loan_amount", "total_outstanding", "avg_interest_rate", "num_loans"
]

X_def = final_features[default_cols]
y_def = final_features['default_flag'].astype(int)

print(f"Features shape: {X_def.shape}")
print(f"Target distribution:")
print(f"  No Default (0): {(y_def == 0).sum():,}")
print(f"  Default (1): {(y_def == 1).sum():,}")
print(f"  Default rate: {y_def.mean():.2%}")

In [0]:
# Split with stratification
X_d_train, X_d_test, y_d_train, y_d_test = train_test_split(
    X_def, y_def, test_size=0.2, stratify=y_def, random_state=42
)

print(f"Training set: {len(X_d_train):,} samples")
print(f"Test set: {len(X_d_test):,} samples")
print(f"Training default rate: {y_d_train.mean():.2%}")
print(f"Test default rate: {y_d_test.mean():.2%}")

In [0]:
print("Training XGBoost Classifier for Default Prediction...")

with mlflow.start_run(run_name="default_risk_predictor") as run:
    # Calculate scale_pos_weight for imbalanced data
    scale_pos_weight = (len(y_d_train) - y_d_train.sum()) / max(1, y_d_train.sum())
    
    # Train model
    xgb_model = xgb.XGBClassifier(
        n_estimators=150,
        max_depth=6,
        learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False
    )
    xgb_model.fit(X_d_train, y_d_train)
    
    # Predictions
    y_d_pred = xgb_model.predict(X_d_test)
    y_d_proba = xgb_model.predict_proba(X_d_test)[:, 1]
    
    # Calculate metrics
    d_acc = accuracy_score(y_d_test, y_d_pred)
    d_prec = precision_score(y_d_test, y_d_pred, zero_division=0)
    d_rec = recall_score(y_d_test, y_d_pred, zero_division=0)
    d_f1 = f1_score(y_d_test, y_d_pred, zero_division=0)
    d_auc = roc_auc_score(y_d_test, y_d_proba)
    
    # Log parameters and metrics
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 150)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("scale_pos_weight", scale_pos_weight)
    mlflow.log_metric("accuracy", d_acc)
    mlflow.log_metric("precision", d_prec)
    mlflow.log_metric("recall", d_rec)
    mlflow.log_metric("f1_score", d_f1)
    mlflow.log_metric("roc_auc", d_auc)
    
    # Log model with signature
    signature = infer_signature(X_d_train, y_d_train)
    mlflow.xgboost.log_model(
        xgb_model,
        "model",
        signature=signature,
        input_example=X_d_train.head(5),
        registered_model_name="hackathon.default.default_risk_predictor"
    )
    
    print("\n" + "="*55)
    print("  DEFAULT PREDICTION RESULTS")
    print("="*55)
    print(f"Accuracy:  {d_acc:.4f}")
    print(f"Precision: {d_prec:.4f}")
    print(f"Recall:    {d_rec:.4f}")
    print(f"F1-Score:  {d_f1:.4f}")
    print(f"ROC-AUC:   {d_auc:.4f}")
    print(f"\nMLflow Run ID: {run.info.run_id}")

In [0]:
# Display confusion matrix
print("\nConfusion Matrix (Default Prediction):")
cm_d = confusion_matrix(y_d_test, y_d_pred)
cm_d_df = pd.DataFrame(
    cm_d, 
    index=["Actual: No Default", "Actual: Default"], 
    columns=["Pred: No Default", "Pred: Default"]
)
display(cm_d_df)

In [0]:
# Display feature importances
print("Feature Importances (Top 15):")
feature_importance_xgb = pd.DataFrame({
    'feature': default_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

display(feature_importance_xgb.head(15))

## 6. Risk Assessment and Explainability

Generate per-user risk assessments with behavioral classifications, default probabilities, and risk tiers.

In [0]:
# Generate predictions for all users
print("Generating risk assessments for all users...")

# Behavioral classification
all_behav_pred = rf.predict(X_behav)

# Default probability
all_default_proba = xgb_model.predict_proba(X_def)[:, 1]

# Add to final features
final_features['behavioral_class'] = all_behav_pred
final_features['default_probability'] = all_default_proba

print(f"Risk assessment completed for {len(final_features):,} users")

In [0]:
# Define risk tier function
def risk_tier(prob):
    if prob < 0.3:
        return "low_risk"
    elif prob < 0.6:
        return "medium_risk"
    else:
        return "high_risk"

# Apply risk tier classification
final_features['risk_tier'] = final_features['default_probability'].apply(risk_tier)

print("\n" + "="*55)
print("  RISK TIER DISTRIBUTION")
print("="*55)
print(final_features['risk_tier'].value_counts().to_string())
print(f"\nHigh Risk Rate: {(final_features['risk_tier'] == 'high_risk').mean():.2%}")

In [0]:
# Display sample of risk assessments
print("\nSample Risk Assessments:")
output_cols = [
    'behavioral_class', 'default_flag', 'default_probability', 'risk_tier',
    'total_loan_amount', 'total_outstanding', 'missed_payment_count',
    'debt_to_income_ratio', 'repayment_ratio'
]

display(final_features[output_cols].head(10))

In [0]:
print("\n" + "="*55)
print("  EXPLAINABILITY: HIGH RISK USER ANALYSIS")
print("="*55)

# Calculate feature importance scores
imp_dict = dict(zip(default_cols, xgb_model.feature_importances_))
pop_mean = X_def.mean()
pop_std = X_def.std()

# Analyze top 5 high-risk users
high_risk_users = final_features[final_features['risk_tier'] == 'high_risk'].head(5)

for uid, row in high_risk_users.iterrows():
    reasons = []
    for feat in default_cols:
        val = X_def.loc[uid, feat]
        m = pop_mean[feat]
        s = pop_std[feat]
        z = (val - m) / s if s > 0 else 0
        score = abs(z) * imp_dict[feat]
        reasons.append({
            "feature": feat,
            "score": score,
            "val": val,
            "mean": m,
            "z": z
        })
    
    top_3 = sorted(reasons, key=lambda x: x["score"], reverse=True)[:3]
    
    print(f"\nUser: {uid}")
    print(f"  Behavioral Class: {row['behavioral_class']}")
    print(f"  Default Probability: {row['default_probability']:.3f}")
    print(f"  Risk Tier: {row['risk_tier']}")
    print(f"  Top 3 Risk Factors:")
    
    for idx, r in enumerate(top_3, 1):
        direction = "Higher" if r['z'] > 0 else "Lower"
        print(f"    {idx}. {r['feature']:<30}: {r['val']:.2f}  ({direction} than avg {r['mean']:.2f})")

## Summary

This notebook successfully:

1. ✅ Loaded transaction and loan data from Unity Catalog tables
2. ✅ Engineered 16 behavioral features from transaction patterns
3. ✅ Engineered 14 debt-related features from loan data
4. ✅ Trained a Random Forest classifier for behavioral categorization
5. ✅ Trained an XGBoost model for default risk prediction
6. ✅ Generated risk assessments with low/medium/high tiers
7. ✅ Provided explainability for high-risk predictions
8. ✅ Logged both models to MLflow with signatures
9. ✅ Registered models to Unity Catalog:
   * `hackathon.default.behavioral_classifier`
   * `hackathon.default.default_risk_predictor`

Both models are now ready for production deployment via Databricks Model Serving.